In [1]:
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"
)

OUT = Path("../data/processed")
OUT.mkdir(parents=True, exist_ok=True)

In [2]:
check_sql = """
SELECT city,
       COUNT(*)                                              AS n_rows,
       MIN(`date`)                                           AS first_date,
       MAX(`date`)                                           AS last_date,
       COUNT(DISTINCT `date`)                                AS n_days,
       SUM(available NOT IN ('t','f') OR available IS NULL)  AS bad_available
FROM raw_calendar
GROUP BY city
"""
raw_check = pd.read_sql(check_sql, engine)
raw_check

,city,n_rows,first_date,last_date,n_days,bad_available
0,Amsterdam,3873380,2026-09-17,2027-09-16,365,0.0
1,Barcelona,5562302,2026-09-17,2027-09-18,367,0.0
2,Berlin,4517243,2026-09-19,2027-09-19,366,0.0
3,Columbus,1080035,2026-09-25,2027-09-24,365,0.0


In [3]:
create_sql = """
CREATE TABLE calendar_clean AS
SELECT c.city,
       c.listing_id,
       STR_TO_DATE(c.`date`, '%Y-%m-%d')                       AS `date`,
       CASE c.available WHEN 't' THEN 1 WHEN 'f' THEN 0 END    AS available,
       CASE WHEN c.minimum_nights REGEXP '^[0-9]+$'
            THEN CAST(c.minimum_nights AS UNSIGNED) END        AS minimum_nights,
       CASE WHEN c.maximum_nights REGEXP '^[0-9]+$'
            THEN CAST(c.maximum_nights AS UNSIGNED) END        AS maximum_nights
FROM raw_calendar c
WHERE EXISTS (SELECT 1 FROM raw_listings l
              WHERE l.city = c.city AND l.id = c.listing_id)
"""

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS calendar_clean"))
    conn.execute(text(create_sql))

with engine.begin() as conn:
    conn.execute(text("ALTER TABLE calendar_clean "
                      "ADD INDEX idx_cc_city_listing (city, listing_id), "
                      "ADD INDEX idx_cc_date (`date`)"))
print("calendar_clean created")

calendar_clean created


In [4]:
clean_check = pd.read_sql("""
SELECT city,
       COUNT(*)                     AS clean_rows,
       COUNT(DISTINCT listing_id)   AS n_listings,
       MIN(`date`)                  AS first_date,
       MAX(`date`)                  AS last_date,
       SUM(`date` IS NULL)          AS null_dates,
       SUM(available IS NULL)       AS null_available,
       ROUND(AVG(available), 3)     AS share_available
FROM calendar_clean
GROUP BY city
""", engine)

# raw vs clean rows -> the difference = orphans removed
cmp = raw_check[["city", "n_rows"]].merge(clean_check[["city", "clean_rows"]], on="city")
cmp["removed"] = cmp["n_rows"] - cmp["clean_rows"]
print(cmp)
clean_check

        city   n_rows  clean_rows  removed
0  Amsterdam  3873380     3873380        0
1  Barcelona  5562302     5562302        0
2     Berlin  4517243     4517243        0
3   Columbus  1080035     1080035        0


,city,clean_rows,n_listings,first_date,last_date,null_dates,null_available,share_available
0,Amsterdam,3873380,10612,2026-09-17,2027-09-16,0.0,0.0,0.276
1,Barcelona,5562302,15236,2026-09-17,2027-09-18,0.0,0.0,0.581
2,Berlin,4517243,12376,2026-09-19,2027-09-19,0.0,0.0,0.442
3,Columbus,1080035,2959,2026-09-25,2027-09-24,0.0,0.0,0.656


In [5]:
# duplicates: same listing + same date twice
dups = pd.read_sql("""
SELECT city, COUNT(*) AS dup_keys FROM (
    SELECT city, listing_id, `date`
    FROM calendar_clean
    GROUP BY city, listing_id, `date`
    HAVING COUNT(*) > 1
) t GROUP BY city
""", engine)
print("duplicate keys:" , "none" if dups.empty else "")
dups

duplicate keys: none


,city,dup_keys


In [6]:
daily = pd.read_sql("""
SELECT city, `date`,
       COUNT(*)                 AS n_listings,
       SUM(available)           AS n_available,
       ROUND(AVG(available), 4) AS share_available
FROM calendar_clean
GROUP BY city, `date`
ORDER BY city, `date`
""", engine)

per_listing = pd.read_sql("""
SELECT city, listing_id,
       COUNT(*)                 AS n_days,
       SUM(available)           AS days_available,
       ROUND(AVG(available), 4) AS share_available,
       MIN(minimum_nights)      AS min_nights_min,
       MAX(minimum_nights)      AS min_nights_max
FROM calendar_clean
GROUP BY city, listing_id
""", engine)

print(daily.shape, per_listing.shape)
daily.head()

(1463, 5) (41183, 7)


,city,date,n_listings,n_available,share_available
0,Amsterdam,2026-09-17,10612,755.0,0.0711
1,Amsterdam,2026-09-18,10612,990.0,0.0933
2,Amsterdam,2026-09-19,10612,1187.0,0.1119
3,Amsterdam,2026-09-20,10612,1915.0,0.1805
4,Amsterdam,2026-09-21,10612,1936.0,0.1824


In [7]:
daily.to_csv(OUT / "calendar_daily_city.csv", index=False)
per_listing.to_csv(OUT / "calendar_listing_summary.csv", index=False)
print("saved:", sorted(p.name for p in OUT.iterdir()))

saved: ['calendar_daily_city.csv', 'calendar_listing_summary.csv', 'listings_clean.csv']
